# **Using ChatGPT or Gemini with Python and LangChain**
In this notebook you will use ChatGPT or Gemini and LangChain to solve and learn about:

Langchain chains
Memory and conversation chains

# **Install OpenAI and LangChain dependencies**

In [4]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12

  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
  Using cached langsmith-0.2.11-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.85-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.84-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.82-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.81-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.80-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.78-py3-none-any.

# **Optional: Install LangChain Google Gemini Dependency**
***Google Gemini API is free (till now). You can get a key here, just need to sign in with your google account. Gemini may not be available fully in EU.***

In [5]:
!pip install langchain-google-genai==2.0.7

  Using cached langchain_google_genai-2.0.7-py3-none-any.whl.metadata (3.6 kB)
Using cached langchain_google_genai-2.0.7-py3-none-any.whl (41 kB)
  Attempting uninstall: langchain-google-genai
    Found existing installation: langchain-google-genai 4.2.4
    Uninstalling langchain-google-genai-4.2.4:
      Successfully uninstalled langchain-google-genai-4.2.4


# **Load OpenAI API Credentials**
***Here we load it from a file so we don't explore the credentials on the internet by mistake***

In [6]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [7]:
import yaml

api_keys_content = """
openai_api_key: "YOUR_OPENAI_API_KEY"
gemini_api_key: "YOUR_GEMINI_API_KEY"
"""

with open('api_keys.yml', 'w') as file:
    file.write(api_keys_content)

print("Created 'api_keys.yml' with placeholder API keys. Please replace them with your actual keys.")

Created 'api_keys.yml' with placeholder API keys. Please replace them with your actual keys.


In [8]:
with open('api_keys.yml', 'r') as file:
    api_creds = yaml.safe_load(file)

api_creds.keys()

dict_keys(['openai_api_key', 'gemini_api_key'])

In [9]:
import os

os.environ['OPENAI_API_KEY'] = api_creds['openai_api_key']

# Optional: Load Gemini API credentials
***Run this section only if you are using Google Gemini***

In [10]:
# import os
# import yaml

# with open('gemini_key.yml', 'r') as file:
#     api_creds = yaml.safe_load(file)

os.environ["GOOGLE_API_KEY"] = api_creds['gemini_api_key']

# **Load Necessary Dependencies and ChatGPT LLM**

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

TypeError: Cannot create a consistent method resolution
order (MRO) for bases ABC, Generic

In [ ]:
model = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.0)

# **Optional: Load Google Gemini LLM**
***Only run the below cell if you don't want to use ChatGPT and want to use Google Gemini***

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_model = ChatGoogleGenerativeAI(model="gemini-2.0-flash",
                                      convert_system_message_to_human=True)

# **Let's look at how to create a basic chain**

In [ ]:
PROMPT = "tell me a joke about {topic}"
prompt = ChatPromptTemplate.from_template(PROMPT)
chain = (
         prompt
         |
         model
)

response = chain.invoke({"topic": "bears"})
print(response.content)

In [ ]:
# same code with gemini
prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
chain = (
         prompt
         |
         gemini_model
)

response = chain.invoke({"topic": "bears"})
print(response.content)

# **Note: Replace model with gemini_model in any of the code below if you want to use Google Gemini instead of ChatGPT**

In [ ]:
# can be used on multiple prompts also
topics = [{'topic': 'AI'}, {'topic': 'Statistics'}]
responses = chain.map().invoke(topics)

In [ ]:
for response in responses:
  print(response.content)
  print('-----')
  print('\n')

# **Basic chains are ad-hoc - No conversation history!**

In [ ]:
prompt = ChatPromptTemplate.from_template("{query}")
basic_chain = (
               prompt
               |
              model
)

In [ ]:
response = basic_chain.invoke({"query" : 'What are the first four colors of the rainbow?'})
print(response.content)

In [ ]:
response = basic_chain.invoke({"query" : 'And the other three?'})
print(response.content) # gives a totally random response

# **Let's learn how to add memory to build a conversation chain**

In [ ]:
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Act as a helpful AI Assistant"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
# k=3 stores last 3 conversations between you and ChatGPT
memory = ConversationBufferWindowMemory(k=3, return_messages=True)

In [ ]:
memory.load_memory_variables({}) # shows the conversation history

In [ ]:
from operator import itemgetter
# creates the conversation chain
chain = (
    RunnablePassthrough.assign(
        history=RunnableLambda(memory.load_memory_variables)
        |
        itemgetter("history")
    )
    |
    prompt
    |
    model
)

In [ ]:
user_input = {'input': 'What are the first four colors of a rainbow'}
response = chain.invoke(user_input)
response.content

In [ ]:
memory.load_memory_variables({})

In [ ]:
# remember to save your conversation to the memory
memory.save_context(user_input, {"output": response.content})
memory.load_memory_variables({}) # remembers the conversation history

In [ ]:
user_input = {'input': 'And the last 3?'}
response = chain.invoke(user_input) # uses history of the past conversation to give a better response
response.content

In [ ]:
memory.save_context(user_input, {"output": response.content})
memory.load_memory_variables({})